# Agent Communication & Hierarchical Systems

Topics:
- **Three communication patterns**: message passing, shared typed fields, blackboard
- **Hierarchical agents**: subgraphs compiled and used as nodes in a parent graph
- `operator.add` on `list[dict]` — accumulating typed data across agents
- `ApprovalDecision` — structured critic/loop-termination with `with_structured_output`

```
Communication patterns:
  Message passing:  researcher → fact_checker → summarizer  (shared message list)
  Shared fields:    collector → analyst → advisor           (typed state fields)
  Blackboard:       drafter ⇆ critic (loop until approved)  (shared workspace)

Hierarchical system:
  ceo → research_team (subgraph)
      → content_team  (subgraph)
      → analysis_team (subgraph)
```

In [ ]:
import json, operator
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict, Annotated
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. Pattern 1 — Message Passing

Agents communicate by reading and appending to a **shared message list**. Each agent sees the entire conversation, which makes downstream agents naturally context-aware.

```
State: { messages: Annotated[list[BaseMessage], add_messages] }

researcher  → appends [RESEARCHER]: findings
fact_checker → reads all messages, appends [FACT-CHECKER]: validation
summarizer  → reads all messages, appends [SUMMARY]: final
```

The `name` field on `AIMessage` (`name="researcher"`) is optional metadata — useful for filtering messages by agent in later nodes.

In [ ]:
class MessagePassingState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    current_phase: str


def create_message_passing_pipeline():

    def researcher(state: MessagePassingState) -> dict:
        response = llm.invoke([
            SystemMessage(content=(
                "You are a researcher. Read the user's question, research it, "
                "and post your findings. Keep it to 2-3 sentences."
            )),
            *state["messages"],
        ])
        return {
            "messages": [AIMessage(content=f"[RESEARCHER]: {response.content}", name="researcher")],
            "current_phase": "fact_checker",
        }

    def fact_checker(state: MessagePassingState) -> dict:
        # Receives the full message list including the researcher's findings
        response = llm.invoke([
            SystemMessage(content=(
                "You are a fact-checker. Read the researcher's findings "
                "in the conversation and validate or challenge them. Keep it to 2-3 sentences."
            )),
            *state["messages"],
        ])
        return {
            "messages": [AIMessage(content=f"[FACT-CHECKER]: {response.content}", name="fact_checker")],
            "current_phase": "summarizer",
        }

    def summarizer(state: MessagePassingState) -> dict:
        # Sees everything: original question + researcher + fact-checker
        response = llm.invoke([
            SystemMessage(content=(
                "You are a summarizer. Read the researcher's findings and "
                "the fact-checker's review. Produce a final, accurate summary. Keep it to 2-3 sentences."
            )),
            *state["messages"],
        ])
        return {
            "messages": [AIMessage(content=f"[SUMMARY]: {response.content}", name="summarizer")],
            "current_phase": "done",
        }

    g = StateGraph(MessagePassingState)
    g.add_node("researcher", researcher)
    g.add_node("fact_checker", fact_checker)
    g.add_node("summarizer", summarizer)
    g.add_edge(START, "researcher")
    g.add_edge("researcher", "fact_checker")
    g.add_edge("fact_checker", "summarizer")
    g.add_edge("summarizer", END)
    return g.compile()


pipeline = create_message_passing_pipeline()
result = pipeline.invoke({
    "messages": [HumanMessage(content="What are the main benefits of renewable energy?")],
    "current_phase": "researcher",
})

for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        print(f"{msg.content}\n")

## 2. Pattern 2 — Shared Typed Fields

Instead of passing everything through messages, agents write to **named state fields**. Downstream agents read exactly the fields they need.

```python
class SharedFieldsState(TypedDict):
    query: str
    raw_data: Annotated[list[dict], operator.add]  # collector appends here
    analysis: str                                   # analyst writes here
    confidence_score: float                         # analyst also writes here
    recommendations: list[str]                      # advisor writes here
```

`Annotated[list[dict], operator.add]` means each node can return `{"raw_data": [...]}` and the lists accumulate — useful when multiple parallel agents all write to the same field.

In [ ]:
class SharedFieldsState(TypedDict):
    query: str
    raw_data: Annotated[list[dict], operator.add]
    analysis: str
    recommendations: list[str]
    confidence_score: float


def create_shared_fields_pipeline():

    def data_collector(state: SharedFieldsState) -> dict:
        """Writes to raw_data field."""
        response = llm.invoke([
            SystemMessage(content=(
                "You are a data collector. Given the query, produce 3 data points "
                "as a JSON array of objects with 'source' and 'finding' keys. "
                "Return ONLY the JSON array, no markdown."
            )),
            HumanMessage(content=state["query"]),
        ])
        try:
            data = json.loads(response.content)
        except json.JSONDecodeError:
            data = [{"source": "llm", "finding": response.content}]
        return {"raw_data": data}

    def analyst(state: SharedFieldsState) -> dict:
        """Reads raw_data, writes analysis and confidence_score."""
        data_summary = json.dumps(state["raw_data"], indent=2)
        response = llm.invoke([
            SystemMessage(content=(
                "You are a data analyst. Analyze the collected data and provide: "
                "1) A brief analysis (2-3 sentences), and "
                "2) A confidence score from 0.0 to 1.0. "
                "Format: ANALYSIS: <text>\nCONFIDENCE: <number>"
            )),
            HumanMessage(content=f"Query: {state['query']}\n\nData:\n{data_summary}"),
        ])
        content = response.content
        analysis = content
        confidence = 0.7
        if "CONFIDENCE:" in content:
            parts = content.split("CONFIDENCE:")
            analysis = parts[0].replace("ANALYSIS:", "").strip()
            try:
                confidence = float(parts[1].strip())
            except ValueError:
                confidence = 0.7
        return {"analysis": analysis, "confidence_score": confidence}

    def advisor(state: SharedFieldsState) -> dict:
        """Reads analysis + confidence_score, writes recommendations."""
        response = llm.invoke([
            SystemMessage(content=(
                "You are a strategic advisor. Based on the analysis and confidence score, "
                "provide 3 actionable recommendations. "
                "Return them as a JSON array of strings. Return ONLY the JSON array, no markdown."
            )),
            HumanMessage(content=(
                f"Query: {state['query']}\n"
                f"Analysis: {state['analysis']}\n"
                f"Confidence: {state['confidence_score']}"
            )),
        ])
        try:
            recs = json.loads(response.content)
        except json.JSONDecodeError:
            recs = [response.content]
        return {"recommendations": recs}

    g = StateGraph(SharedFieldsState)
    g.add_node("data_collector", data_collector)
    g.add_node("analyst", analyst)
    g.add_node("advisor", advisor)
    g.add_edge(START, "data_collector")
    g.add_edge("data_collector", "analyst")
    g.add_edge("analyst", "advisor")
    g.add_edge("advisor", END)
    return g.compile()


pipeline = create_shared_fields_pipeline()
result = pipeline.invoke({
    "query": "Should a small business invest in AI automation in 2026?",
    "raw_data": [],
    "analysis": "",
    "recommendations": [],
    "confidence_score": 0.0,
})

print(f"Data points collected: {len(result['raw_data'])}")
for d in result["raw_data"]:
    print(f"  [{d.get('source', 'N/A')}] {d.get('finding', 'N/A')[:80]}...")

print(f"\nAnalysis: {result['analysis'][:200]}...")
print(f"Confidence: {result['confidence_score']}")
print("\nRecommendations:")
for i, rec in enumerate(result["recommendations"], 1):
    print(f"  {i}. {rec}")

## 3. Pattern 3 — Blackboard (Iterative Refinement)

The **blackboard** pattern combines shared state + messages + a refinement loop. A drafter writes, a critic reviews, and the loop continues until approved.

```
START → drafter → critic ──► END        (if approved)
                  critic ──► drafter    (if rejected, loop back)
```

Key pattern: using `with_structured_output(ApprovalDecision)` for the critic gives a reliable `approved: bool` that the routing function can use directly — no regex parsing on LLM text.

In [ ]:
class BlackboardState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    topic: str
    drafts: Annotated[list[str], operator.add]    # accumulates each draft
    critiques: Annotated[list[str], operator.add] # accumulates each critique
    iteration: int
    is_approved: bool


def create_blackboard_system():

    class ApprovalDecision(BaseModel):
        approved: bool = Field(description="Whether the draft is good enough")
        feedback: str = Field(description="Specific feedback if not approved")

    critic_llm = llm.with_structured_output(ApprovalDecision)

    def drafter(state: BlackboardState) -> dict:
        """Reads critiques from blackboard, writes improved draft."""
        context_parts = [f"Topic: {state['topic']}"]
        if state["drafts"]:
            context_parts.append(f"Previous draft: {state['drafts'][-1]}")
        if state["critiques"]:
            context_parts.append(f"Feedback to address: {state['critiques'][-1]}")

        response = llm.invoke([
            SystemMessage(content=(
                "You are a skilled writer. Write or revise a short paragraph "
                "(3-4 sentences). If there's feedback, directly address it in your revision."
            )),
            HumanMessage(content="\n".join(context_parts)),
        ])
        return {
            "drafts": [response.content],
            "messages": [AIMessage(content=f"[DRAFTER iter {state['iteration']+1}]: {response.content}", name="drafter")],
            "iteration": state["iteration"] + 1,
        }

    def critic(state: BlackboardState) -> dict:
        """Reviews the latest draft; approves or sends feedback."""
        latest_draft = state["drafts"][-1] if state["drafts"] else "No draft yet"

        decision = critic_llm.invoke([
            SystemMessage(content=(
                "You are a strict editor. Review the draft for clarity, accuracy, and engagement. "
                "Approve ONLY if it's genuinely good. If iteration >= 3, be more lenient."
            )),
            HumanMessage(content=(
                f"Topic: {state['topic']}\n"
                f"Iteration: {state['iteration']}\n"
                f"Draft: {latest_draft}"
            )),
        ])

        # Force approval after 3 iterations to prevent infinite loops
        approved = decision.approved or state["iteration"] >= 3

        result = {
            "is_approved": approved,
            "messages": [AIMessage(
                content=f"[CRITIC]: {'APPROVED' if approved else 'REVISION NEEDED'} — {decision.feedback}",
                name="critic",
            )],
        }
        if not approved:
            result["critiques"] = [decision.feedback]
        return result

    def route_after_critic(state: BlackboardState) -> Literal["drafter", "end"]:
        return "end" if state["is_approved"] else "drafter"

    g = StateGraph(BlackboardState)
    g.add_node("drafter", drafter)
    g.add_node("critic", critic)
    g.add_edge(START, "drafter")
    g.add_edge("drafter", "critic")
    g.add_conditional_edges("critic", route_after_critic, {"drafter": "drafter", "end": END})
    return g.compile()


blackboard = create_blackboard_system()
result = blackboard.invoke({
    "messages": [],
    "topic": "Why LangGraph is great for building multi-agent systems",
    "drafts": [],
    "critiques": [],
    "iteration": 0,
    "is_approved": False,
})

print(f"Total iterations: {result['iteration']}  |  Approved: {result['is_approved']}")
print("\nConversation:")
for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        print(f"\n{msg.content}")
print(f"\nFinal draft:\n{result['drafts'][-1]}")

## 4. Hierarchical Agents — Subgraphs as Nodes

A **hierarchical system** composes multiple compiled graphs: each department is built as an independent subgraph, compiled, then added as a **single node** in the parent graph.

```python
research_team = build_research_team().compile()  # compiled subgraph
parent.add_node("research_team", research_team)  # used as a node
```

The parent graph's CEO supervisor routes to the right department. Each department subgraph handles its own internal logic (possibly fan-out, its own loops, etc.) and returns `final_answer` to the parent state when done.

**Shared state**: all levels use the same `TeamState` TypedDict, so state flows naturally between parent and subgraph.

In [ ]:
class TeamState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    final_answer: str


# ---- Department 1: Research Team ----
def build_research_team() -> StateGraph:
    """Two parallel researchers + a lead who synthesizes."""

    def web_researcher(state: TeamState) -> dict:
        query = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "")
        response = llm.invoke([
            SystemMessage(content="You are a web researcher. Find key facts. Provide 3-4 bullet points."),
            HumanMessage(content=query),
        ])
        return {"messages": [AIMessage(content=f"[WEB RESEARCHER]: {response.content}", name="web_researcher")]}

    def paper_reviewer(state: TeamState) -> dict:
        query = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "")
        response = llm.invoke([
            SystemMessage(content="You are an academic reviewer. Provide technical depth. 3-4 bullet points."),
            HumanMessage(content=query),
        ])
        return {"messages": [AIMessage(content=f"[PAPER REVIEWER]: {response.content}", name="paper_reviewer")]}

    def research_lead(state: TeamState) -> dict:
        response = llm.invoke([
            SystemMessage(content="You are the research lead. Synthesize the findings into one short paragraph."),
            *state["messages"],
        ])
        return {
            "messages": [AIMessage(content=f"[RESEARCH LEAD]: {response.content}", name="research_lead")],
            "final_answer": response.content,
        }

    g = StateGraph(TeamState)
    g.add_node("web_researcher", web_researcher)
    g.add_node("paper_reviewer", paper_reviewer)
    g.add_node("research_lead", research_lead)
    # Fan-out: both researchers work in parallel
    g.add_edge(START, "web_researcher")
    g.add_edge(START, "paper_reviewer")
    # Fan-in: both feed research_lead
    g.add_edge("web_researcher", "research_lead")
    g.add_edge("paper_reviewer", "research_lead")
    g.add_edge("research_lead", END)
    return g


# ---- Department 2: Content Team ----
def build_content_team() -> StateGraph:
    def content_writer(state: TeamState) -> dict:
        response = llm.invoke([
            SystemMessage(content="You are a skilled content writer. Write a clear, engaging short piece (one paragraph)."),
            *state["messages"],
        ])
        return {"messages": [AIMessage(content=f"[WRITER]: {response.content}", name="content_writer")]}

    def content_editor(state: TeamState) -> dict:
        response = llm.invoke([
            SystemMessage(content="You are a content editor. Polish the writer's draft — return the polished version only."),
            *state["messages"],
        ])
        return {
            "messages": [AIMessage(content=f"[EDITOR]: {response.content}", name="content_editor")],
            "final_answer": response.content,
        }

    g = StateGraph(TeamState)
    g.add_node("writer", content_writer)
    g.add_node("editor", content_editor)
    g.add_edge(START, "writer")
    g.add_edge("writer", "editor")
    g.add_edge("editor", END)
    return g


# ---- Department 3: Analysis Team ----
def build_analysis_team() -> StateGraph:
    def data_analyst(state: TeamState) -> dict:
        response = llm.invoke([
            SystemMessage(content="You are a data analyst. Provide 3-4 data-driven insights."),
            *state["messages"],
        ])
        return {"messages": [AIMessage(content=f"[DATA ANALYST]: {response.content}", name="data_analyst")]}

    def strategy_advisor(state: TeamState) -> dict:
        response = llm.invoke([
            SystemMessage(content="You are a strategy advisor. Based on the analysis, provide 3 actionable recommendations."),
            *state["messages"],
        ])
        return {
            "messages": [AIMessage(content=f"[STRATEGY ADVISOR]: {response.content}", name="strategy_advisor")],
            "final_answer": response.content,
        }

    g = StateGraph(TeamState)
    g.add_node("data_analyst", data_analyst)
    g.add_node("strategy_advisor", strategy_advisor)
    g.add_edge(START, "data_analyst")
    g.add_edge("data_analyst", "strategy_advisor")
    g.add_edge("strategy_advisor", END)
    return g

In [ ]:
def create_hierarchical_system():
    """CEO supervisor routes to compiled department subgraphs."""

    # Compile each department into a reusable subgraph
    research_team = build_research_team().compile()
    content_team  = build_content_team().compile()
    analysis_team = build_analysis_team().compile()

    class DepartmentRoute(BaseModel):
        department: Literal["research", "content", "analysis"] = Field(
            description="Which department should handle this request"
        )
        reasoning: str

    router_llm = llm.with_structured_output(DepartmentRoute)

    def ceo_supervisor(state: TeamState) -> dict:
        decision = router_llm.invoke([
            SystemMessage(content=(
                "You are the CEO supervisor. Route the request to the right department:\n"
                "- research: Fact-finding, investigation, technical deep-dives\n"
                "- content: Writing, blog posts, marketing copy, summaries\n"
                "- analysis: Data analysis, strategy, business decisions"
            )),
            *state["messages"],
        ])
        return {
            "messages": [AIMessage(
                content=f"[CEO]: Routing to {decision.department} — {decision.reasoning}",
                name="ceo",
            )]
        }

    def route_to_department(state: TeamState) -> str:
        """Read the CEO's last message to determine which department to activate."""
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage) and msg.name == "ceo":
                content = msg.content.lower()
                if "research" in content:  return "research_team"
                if "content"  in content:  return "content_team"
                if "analysis" in content:  return "analysis_team"
        return "research_team"  # default

    parent = StateGraph(TeamState)
    parent.add_node("ceo", ceo_supervisor)
    parent.add_node("research_team", research_team)  # compiled subgraph as node
    parent.add_node("content_team",  content_team)
    parent.add_node("analysis_team", analysis_team)

    parent.add_edge(START, "ceo")
    parent.add_conditional_edges(
        "ceo", route_to_department,
        {"research_team": "research_team", "content_team": "content_team", "analysis_team": "analysis_team"},
    )
    for node in ["research_team", "content_team", "analysis_team"]:
        parent.add_edge(node, END)

    return parent.compile()


system = create_hierarchical_system()

queries = [
    "What are the latest trends in large language models?",
    "Write a short blog introduction about AI agents",
    "Should my startup invest in building AI features this year?",
]

for query in queries:
    print(f"Query: {query}")
    result = system.invoke({"messages": [HumanMessage(content=query)], "final_answer": ""})
    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.name == "ceo":
            print(f"  {msg.content}")
    print(f"  Final: {result['final_answer'][:200]}...\n")